# KL-IG Evaluation Notebook

**Methods compared:**
| Method | Baseline | Path |
|--------|----------|------|
| Vanilla IG | Zero (black image) | Pixel-space linear |
| Blur IG | Gaussian-blurred image | Pixel-space linear |
| KL-IG | N(0, I) prior | Distribution-space (mu, logvar) |

**Evaluation metrics:**
| Category | Metric | What it measures |
|----------|--------|-----------------|
| **Faithfulness** | Insertion / Deletion | Does masking by attribution order change predictions? |
| **Faithfulness** | ROAR-lite | Are top-attributed features truly important? |
| **Faithfulness** | Completeness: sum(attr) ~ delta_f | Do attributions sum to the output difference? |
| **Robustness** | Perturbation sigma | Attribution stability under input noise |
| **Robustness** | Variance sigma^2 | Run-to-run variance of attributions |
| **Sensitivity** | Occlusion correlation | Agreement with occlusion importance |
| **Sensitivity** | Sparsity (Gini) | How concentrated are attributions? |

## 0. Environment Setup (Colab)

In [ ]:
!git clone https://github.com/Shameen5375/KLIG_V1.git 2>/dev/null || echo "Repo already cloned"
%cd KLIG_V1

In [ ]:
!pip install -q captum

## 1. Imports & Device

In [ ]:
import sys, math, warnings
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet50_Weights, resnet50
from scipy import stats
from tqdm.notebook import tqdm

ROOT = Path.cwd()
if not (ROOT / "klig").exists():
    ROOT = ROOT / "infocube-main"
sys.path.append(str(ROOT))

from klig.image.attribution import ImageAttributor
from klig.image.stopping import find_sigma_stop
from klig.compare.captum_baselines import run_ig, _absmax_collapse
from klig.image.viz import _attr_to_rgb
from captum.attr import IntegratedGradients

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Root:   {ROOT}")

## 2. Download Test Images & Configure

In [ ]:
!mkdir -p images

# Download diverse images (different classes so predictions differ)
!wget -q "https://upload.wikimedia.org/wikipedia/commons/2/26/YellowLabradorLooking_new.jpg" -O images/dog.jpg
!wget -q "https://upload.wikimedia.org/wikipedia/commons/4/4d/Cat_November_2010-1a.jpg"       -O images/cat.jpg
!wget -q "https://upload.wikimedia.org/wikipedia/commons/thumb/4/45/GuitareClassique5.png/220px-GuitareClassique5.png" -O images/guitar.png
!wget -q "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg" -O images/tabby.jpg

!ls -lh images/

In [ ]:
# ── Collect image paths ──
IMAGE_PATHS = sorted(
    list((ROOT / "images").glob("*.jpg")) +
    list((ROOT / "images").glob("*.png"))
)
assert len(IMAGE_PATHS) > 0, "No images found in images/ folder!"
print(f"Found {len(IMAGE_PATHS)} images:")
for p in IMAGE_PATHS:
    print(f"  {p.name}")

# ── Hyperparameters ──
N_STEPS       = 50        # KL-IG integration steps
N_SAMPLES     = 10        # KL-IG MC samples per step
SIGMA_FINAL   = 1 / 256   # KL-IG default final sigma
ADAPTIVE_SIGMA = True     # use binary-search sigma_stop

IG_STEPS      = 50        # Vanilla / Blur IG steps
BLUR_SIGMA    = 16.0      # Gaussian blur sigma for Blur IG
BLUR_KERNEL   = 51        # kernel size (odd)

N_INSERTION_STEPS   = 100
N_ROBUSTNESS_RUNS   = 5
PERTURBATION_SIGMAS = [0.01, 0.02, 0.05, 0.1, 0.2]
OCCLUSION_PATCH     = 14
OCCLUSION_STRIDE    = 7
CLIP_PCT            = 99.0

# ── ImageNet normalization ──
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
TRANSFORM = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

COLORS = {"Vanilla IG": "#7B68EE", "Blur IG": "#E07B39", "KL-IG": "#2E8B57"}
print("\nConfig OK.")

## 3. Helper Functions

In [ ]:
# ── Model ──
def load_model():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights).to(DEVICE).eval()
    return model, weights.meta["categories"]

# ── Image I/O ──
def load_image(path):
    """Load ONE image file -> (1, C, H, W) tensor."""
    img = Image.open(str(path)).convert("RGB")
    return TRANSFORM(img).unsqueeze(0).to(DEVICE)

def denormalize(x):
    """Undo ImageNet norm -> [0,1] for display."""
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(-1, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=x.device).view(-1, 1, 1)
    if x.dim() == 4:
        mean, std = mean.unsqueeze(0), std.unsqueeze(0)
    return (x * std + mean).clamp(0, 1)

def predict_topk(model, x, k=5):
    with torch.no_grad():
        probs = model(x).softmax(-1)[0]
        top_p, top_i = probs.topk(k)
    cats = ResNet50_Weights.IMAGENET1K_V2.meta["categories"]
    return [(cats[int(i)], float(p)) for i, p in zip(top_i, top_p)], top_i.tolist()

# ── Blur baseline ──
def make_blur_baseline(x, kernel_size=BLUR_KERNEL, sigma=BLUR_SIGMA):
    coords = torch.arange(kernel_size, dtype=torch.float32, device=x.device) - kernel_size // 2
    k1d = torch.exp(-0.5 * (coords / sigma) ** 2)
    k1d = k1d / k1d.sum()
    kh = k1d.view(1, 1, -1, 1).expand(3, -1, -1, -1)
    kw = k1d.view(1, 1, 1, -1).expand(3, -1, -1, -1)
    pad = kernel_size // 2
    out = F.conv2d(x, kh, padding=(pad, 0), groups=3)
    return F.conv2d(out, kw, padding=(0, pad), groups=3)

# ── Three attribution methods ──
def compute_vanilla_ig(model, x, target):
    return run_ig(model, x, target, n_steps=IG_STEPS)

def compute_blur_ig(model, x, target):
    baseline = make_blur_baseline(x)
    return run_ig(model, x, target, n_steps=IG_STEPS, baseline=baseline)

def compute_klig(model, x, target):
    sf = SIGMA_FINAL
    if ADAPTIVE_SIGMA:
        sf = max(find_sigma_stop(model, x, target=target, tau=0.95), 1.0 / 256.0)
    attr = ImageAttributor(model=model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                           sigma_final=sf, device=DEVICE)
    return attr.attribute(x, target=target, show_progress=False)

def compute_all(model, x, target):
    """Returns (dict of (H,W) maps, klig_result)."""
    v = compute_vanilla_ig(model, x, target)
    b = compute_blur_ig(model, x, target)
    k = compute_klig(model, x, target)
    maps = {"Vanilla IG": v, "Blur IG": b, "KL-IG": k.attr_map("absmax")}
    return maps, k

print("Helpers ready.")

## 4. Load Model & All Images

In [ ]:
model, imagenet_labels = load_model()
print(f"Model: ResNet50 ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)\n")

# Load each image independently and predict
images = {}
for path in IMAGE_PATHS:
    x_i = load_image(path)                       # (1,3,224,224) -- loaded from THIS file
    preds_i, idx_i = predict_topk(model, x_i)    # predictions for THIS image
    images[path.stem] = dict(path=path, x=x_i, preds=preds_i,
                             top_idx=idx_i, target=idx_i[0])

# Show gallery with per-image predictions
n = len(images)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1: axes = [axes]
for ax, (name, d) in zip(axes, images.items()):
    ax.imshow(denormalize(d["x"][0]).cpu().permute(1, 2, 0).numpy())
    ax.set_title(f"{name}\n{d['preds'][0][0]} ({d['preds'][0][1]:.1%})", fontsize=9)
    ax.axis("off")
plt.suptitle("Loaded Images & Top-1 Predictions", fontweight="bold")
plt.tight_layout(); plt.show()

# Print full top-5 per image
for name, d in images.items():
    print(f"\n{name} -> target class {d['target']}:")
    for lab, p in d["preds"]:
        print(f"  {lab}: {p:.4f}")

## 5. Compute Attributions (All Images x All Methods)

In [ ]:
results = {}   # {image_name: {"maps": {method: (H,W)}, "klig": klig_result, ...}}

for name, d in images.items():
    print(f"[{name}] computing attributions...")
    maps, klig_res = compute_all(model, d["x"], d["target"])
    results[name] = {"maps": maps, "klig": klig_res, **d}
    print(f"  KL-IG completeness = {klig_res._r.completeness_check():.4f}")

# ── Grid: rows = images, cols = [original, Vanilla IG, Blur IG, KL-IG] ──
n_imgs = len(results)
fig, axes = plt.subplots(n_imgs, 4, figsize=(16, 4 * n_imgs), squeeze=False)

for row, (name, r) in enumerate(results.items()):
    axes[row][0].imshow(denormalize(r["x"][0]).cpu().permute(1, 2, 0).numpy())
    axes[row][0].set_title(f"{name}\n{r['preds'][0][0]}", fontsize=9)
    axes[row][0].axis("off")
    for col, (method, amap) in enumerate(r["maps"].items()):
        axes[row][col+1].imshow(_attr_to_rgb(amap, clip_percentile=CLIP_PCT))
        axes[row][col+1].set_title(method if row == 0 else "", fontsize=10)
        axes[row][col+1].axis("off")

fig.suptitle("Attribution Maps: All Images x All Methods", fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()

---
## 6. Faithfulness: Insertion / Deletion Curves

In [ ]:
def insertion_deletion(model, x, attr_map, target, n_steps=N_INSERTION_STEPS):
    H, W = attr_map.shape
    n_pix = H * W
    order = attr_map.detach().cpu().abs().view(-1).argsort(descending=True)
    substrate = make_blur_baseline(x)
    pps = max(1, n_pix // n_steps)

    ins_img, del_img = substrate.clone(), x.clone()
    ins_scores, del_scores = [], []

    with torch.no_grad():
        ins_scores.append(model(ins_img).softmax(-1)[0, target].item())
        del_scores.append(model(del_img).softmax(-1)[0, target].item())
        for step in range(1, n_steps + 1):
            s, e = (step-1)*pps, min(step*pps, n_pix)
            if s >= n_pix:
                ins_scores.append(ins_scores[-1]); del_scores.append(del_scores[-1]); continue
            idx = order[s:e]
            hi, wi = idx // W, idx % W
            ins_img[0, :, hi, wi] = x[0, :, hi, wi]
            del_img[0, :, hi, wi] = substrate[0, :, hi, wi]
            ins_scores.append(model(ins_img).softmax(-1)[0, target].item())
            del_scores.append(model(del_img).softmax(-1)[0, target].item())

    ins, dl = np.array(ins_scores), np.array(del_scores)
    return dict(insertion=ins, deletion=dl,
                ins_auc=np.trapezoid(ins, dx=1/n_steps),
                del_auc=np.trapezoid(dl, dx=1/n_steps))

# Run for every image
id_all = {}
for name, r in results.items():
    id_all[name] = {}
    for method, amap in r["maps"].items():
        id_all[name][method] = insertion_deletion(model, r["x"], amap, r["target"])

# Plot per image
frac = np.linspace(0, 1, N_INSERTION_STEPS + 1)
for name in results:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for method, res in id_all[name].items():
        axes[0].plot(frac, res["insertion"], label=f'{method} (AUC={res["ins_auc"]:.3f})',
                     color=COLORS[method], lw=2)
        axes[1].plot(frac, res["deletion"], label=f'{method} (AUC={res["del_auc"]:.3f})',
                     color=COLORS[method], lw=2)
    axes[0].set(title=f"{name}: Insertion (higher=better)", xlabel="frac inserted", ylabel="P(target)")
    axes[1].set(title=f"{name}: Deletion (lower=better)",  xlabel="frac deleted",  ylabel="P(target)")
    for a in axes: a.legend(fontsize=8); a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## 7. Faithfulness: ROAR-lite

In [ ]:
def roar_lite(model, x, attr_map, target,
              pcts=(0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.90, 1.0)):
    H, W = attr_map.shape
    order = attr_map.detach().cpu().abs().view(-1).argsort(descending=True)
    sub = make_blur_baseline(x)
    with torch.no_grad():
        clean = model(x)[0, target].item()
    drops, probs = [], []
    with torch.no_grad():
        for p in pcts:
            m = x.clone()
            k = int(p * H * W)
            if k > 0:
                idx = order[:k]; m[0, :, idx // W, idx % W] = sub[0, :, idx // W, idx % W]
            logit = model(m)[0, target].item()
            drops.append(clean - logit)
            probs.append(model(m).softmax(-1)[0, target].item())
    return dict(pcts=np.array(pcts), drops=np.array(drops), probs=np.array(probs))

# Run & plot per image
for name, r in results.items():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for method, amap in r["maps"].items():
        rr = roar_lite(model, r["x"], amap, r["target"])
        axes[0].plot(rr["pcts"]*100, rr["drops"], "o-", label=method, color=COLORS[method], lw=2, ms=4)
        axes[1].plot(rr["pcts"]*100, rr["probs"], "o-", label=method, color=COLORS[method], lw=2, ms=4)
    axes[0].set(title=f"{name}: Logit Drop (higher=faithful)", xlabel="% removed", ylabel="logit drop")
    axes[1].set(title=f"{name}: P(target) remaining",         xlabel="% removed", ylabel="P(target)")
    for a in axes: a.legend(fontsize=8); a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## 8. Faithfulness: Completeness Axiom

In [ ]:
def completeness(model, x, target, klig_result):
    ig = IntegratedGradients(model)
    out = {}
    with torch.no_grad():
        f_x = model(x)[0, target].item()

    # Vanilla IG
    zero = torch.zeros_like(x)
    with torch.no_grad(): f_zero = model(zero)[0, target].item()
    attr_v = ig.attribute(x, baselines=zero, target=target, n_steps=IG_STEPS, method="gausslegendre")
    s_v = attr_v.detach().sum().item(); d_v = f_x - f_zero
    out["Vanilla IG"] = dict(sum_attr=s_v, delta_f=d_v, rel_err=abs(s_v-d_v)/(abs(d_v)+1e-10))

    # Blur IG
    blur = make_blur_baseline(x)
    with torch.no_grad(): f_blur = model(blur)[0, target].item()
    attr_b = ig.attribute(x, baselines=blur, target=target, n_steps=IG_STEPS, method="gausslegendre")
    s_b = attr_b.detach().sum().item(); d_b = f_x - f_blur
    out["Blur IG"] = dict(sum_attr=s_b, delta_f=d_b, rel_err=abs(s_b-d_b)/(abs(d_b)+1e-10))

    # KL-IG
    s_k = klig_result._r.completeness_check()
    with torch.no_grad():
        noise = torch.randn(64, *x.squeeze(0).shape, device=DEVICE)
        f_noise = model(noise)[:, target].mean().item()
    d_k = f_x - f_noise
    out["KL-IG"] = dict(sum_attr=s_k, delta_f=d_k, rel_err=abs(s_k-d_k)/(abs(d_k)+1e-10))
    return out

# Run across all images
comp_all = {}
for name, r in results.items():
    comp_all[name] = completeness(model, r["x"], r["target"], r["klig"])

# Aggregate bar chart
methods = ["Vanilla IG", "Blur IG", "KL-IG"]
avg_err = {m: np.mean([comp_all[n][m]["rel_err"] for n in comp_all]) for m in methods}

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(methods, [avg_err[m]*100 for m in methods],
              color=[COLORS[m] for m in methods], edgecolor="black", lw=0.5)
ax.set_ylabel("Relative Completeness Error (%)")
ax.set_title("Completeness: sum(attr) vs delta_f  (lower=better)", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for b, m in zip(bars, methods):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f"{avg_err[m]*100:.2f}%",
            ha="center", fontsize=10)
plt.tight_layout(); plt.show()

# Per-image table
for name in comp_all:
    print(f"\n{name}:")
    print(f"  {'Method':<12} {'sum(attr)':>10} {'delta_f':>10} {'rel_err':>10}")
    for m in methods:
        c = comp_all[name][m]
        print(f"  {m:<12} {c['sum_attr']:>10.4f} {c['delta_f']:>10.4f} {c['rel_err']:>9.2%}")

---
## 9. Robustness: Perturbation Sensitivity

In [ ]:
def perturbation_robustness(model, x, target, clean_maps, sigmas=PERTURBATION_SIGMAS, n_runs=3):
    """Spearman correlation of noisy-input attribution vs clean attribution."""
    out = {m: {} for m in clean_maps}
    for sigma in sigmas:
        for method in clean_maps:
            corrs = []
            for _ in range(n_runs):
                x_noisy = x + sigma * torch.randn_like(x)
                if method == "Vanilla IG":   nm = compute_vanilla_ig(model, x_noisy, target)
                elif method == "Blur IG":    nm = compute_blur_ig(model, x_noisy, target)
                else:                        nm = compute_klig(model, x_noisy, target).attr_map("absmax")
                corr, _ = stats.spearmanr(clean_maps[method].detach().cpu().numpy().ravel(),
                                          nm.detach().cpu().numpy().ravel())
                corrs.append(corr)
            out[method][sigma] = np.mean(corrs)
    return out

# Use first image for robustness (expensive)
first_name = list(results.keys())[0]
r0 = results[first_name]
print(f"Perturbation robustness on: {first_name}")
perturb = perturbation_robustness(model, r0["x"], r0["target"], r0["maps"])

fig, ax = plt.subplots(figsize=(9, 5))
for method in perturb:
    sigs = sorted(perturb[method])
    ax.plot(sigs, [perturb[method][s] for s in sigs], "o-",
            label=method, color=COLORS[method], lw=2, ms=6)
ax.set(xlabel="Perturbation sigma", ylabel="Spearman corr with clean",
       title="Perturbation Robustness (higher=more stable)", ylim=(0, 1.05))
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Robustness: Run-to-Run Variance

In [ ]:
def variance_stability(model, x, target, n_runs=N_ROBUSTNESS_RUNS, eps=1e-3):
    stacks = {"Vanilla IG": [], "Blur IG": [], "KL-IG": []}
    for _ in range(n_runs):
        xe = x + eps * torch.randn_like(x)  # tiny jitter for deterministic methods
        stacks["Vanilla IG"].append(compute_vanilla_ig(model, xe, target).detach().cpu().numpy())
        stacks["Blur IG"].append(compute_blur_ig(model, xe, target).detach().cpu().numpy())
        stacks["KL-IG"].append(compute_klig(model, x, target).attr_map("absmax").detach().cpu().numpy())
    out = {}
    for m, arrs in stacks.items():
        s = np.stack(arrs)
        v = s.var(axis=0)
        out[m] = dict(mean_var=v.mean(), max_var=v.max(), var_map=v)
    return out

print(f"Variance stability on: {first_name}  ({N_ROBUSTNESS_RUNS} runs)")
var_res = variance_stability(model, r0["x"], r0["target"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (m, v) in enumerate(var_res.items()):
    im = axes[i].imshow(v["var_map"], cmap="hot", interpolation="nearest")
    axes[i].set_title(f"{m}\nmean_var={v['mean_var']:.2e}", fontsize=10); axes[i].axis("off")
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)
fig.suptitle("Pixel-wise Variance (lower=more stable)", fontweight="bold")
plt.tight_layout(); plt.show()

---
## 11. Sensitivity: Occlusion Correlation

In [ ]:
def occlusion_map(model, x, target, ps=OCCLUSION_PATCH, stride=OCCLUSION_STRIDE):
    _, C, H, W = x.shape
    with torch.no_grad(): f0 = model(x)[0, target].item()
    hs = list(range(0, H - ps + 1, stride))
    ws = list(range(0, W - ps + 1, stride))
    om = np.zeros((len(hs), len(ws)))
    with torch.no_grad():
        for i, h in enumerate(hs):
            for j, w in enumerate(ws):
                m = x.clone(); m[0, :, h:h+ps, w:w+ps] = 0
                om[i, j] = f0 - model(m)[0, target].item()
    return om

def attr_to_grid(attr_map, ps, stride, H, W):
    a = attr_map.detach().cpu().abs().numpy()
    hs = list(range(0, H - ps + 1, stride))
    ws = list(range(0, W - ps + 1, stride))
    g = np.zeros((len(hs), len(ws)))
    for i, h in enumerate(hs):
        for j, w in enumerate(ws):
            g[i, j] = a[h:h+ps, w:w+ps].mean()
    return g

# Run per image
occ_corrs = defaultdict(dict)
for name, r in results.items():
    om = occlusion_map(model, r["x"], r["target"])
    _, _, H, W = r["x"].shape
    for method, amap in r["maps"].items():
        ag = attr_to_grid(amap, OCCLUSION_PATCH, OCCLUSION_STRIDE, H, W)
        rho, _ = stats.spearmanr(om.ravel(), ag.ravel())
        occ_corrs[name][method] = rho

# Aggregate bar chart
avg_occ = {m: np.mean([occ_corrs[n][m] for n in occ_corrs]) for m in methods}
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(methods, [avg_occ[m] for m in methods],
              color=[COLORS[m] for m in methods], edgecolor="black", lw=0.5)
ax.set_ylabel("Spearman rho"); ax.set_title("Occlusion Correlation (higher=better)", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for b, m in zip(bars, methods):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f"{avg_occ[m]:.3f}", ha="center", fontsize=10)
plt.tight_layout(); plt.show()

# Per-image
for name in occ_corrs:
    print(f"{name}: " + "  ".join(f"{m}={occ_corrs[name][m]:.3f}" for m in methods))

## 12. Sensitivity: Sparsity (Gini + Lorenz)

In [ ]:
def gini(values):
    v = np.sort(np.abs(values.ravel()))
    n = len(v)
    if n == 0 or v.sum() == 0: return 0.0
    return (2 * np.sum(np.arange(1, n+1) * v) / (n * v.sum())) - (n+1) / n

def top_k_mass(attr, k_pct):
    f = np.abs(attr.detach().cpu().numpy().ravel())
    k = max(1, int(k_pct / 100 * len(f)))
    return np.sort(f)[::-1][:k].sum() / (f.sum() + 1e-12)

# Compute across all images
gini_all = defaultdict(list)
for name, r in results.items():
    for method, amap in r["maps"].items():
        gini_all[method].append(gini(amap.detach().cpu().numpy()))

avg_gini = {m: np.mean(gini_all[m]) for m in methods}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(methods, [avg_gini[m] for m in methods],
                   color=[COLORS[m] for m in methods], edgecolor="black", lw=0.5)
axes[0].set_ylabel("Gini Coefficient")
axes[0].set_title("Sparsity (higher=more concentrated)", fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)
for b, m in zip(bars, methods):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.003,
                 f"{avg_gini[m]:.4f}", ha="center", fontsize=10)

# Lorenz curves (first image)
r0 = results[first_name]
for method, amap in r0["maps"].items():
    f = np.sort(np.abs(amap.detach().cpu().numpy().ravel()))
    cum = np.cumsum(f) / f.sum()
    axes[1].plot(np.linspace(0, 1, len(cum)), cum,
                 label=f"{method} (G={gini(amap.detach().cpu().numpy()):.3f})",
                 color=COLORS[method], lw=2)
axes[1].plot([0,1], [0,1], "k--", alpha=0.3, label="equality")
axes[1].set(xlabel="Fraction of pixels", ylabel="Cumulative |attr|",
            title=f"Lorenz Curves ({first_name})")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3); axes[1].set_aspect("equal")
plt.tight_layout(); plt.show()

---
## 13. OOD Baseline Analysis

In [ ]:
def analyse_baselines(model, x, target):
    max_H = np.log(1000)
    with torch.no_grad():
        # Zero baseline
        z = torch.zeros_like(x)
        pz = model(z).softmax(-1)[0]
        Hz = -(pz * pz.log().clamp(min=-100)).sum().item()
        # Blur baseline
        bl = make_blur_baseline(x)
        pb = model(bl).softmax(-1)[0]
        Hb = -(pb * pb.log().clamp(min=-100)).sum().item()
        # N(0,1)
        noise = torch.randn(50, *x.squeeze(0).shape, device=DEVICE)
        pn = model(noise).softmax(-1).mean(0)
        Hn = -(pn * pn.log().clamp(min=-100)).sum().item()
    return {
        "Zero":  dict(p_target=pz[target].item(), entropy_ratio=Hz/max_H),
        "Blur":  dict(p_target=pb[target].item(), entropy_ratio=Hb/max_H),
        "N(0,1)": dict(p_target=pn[target].item(), entropy_ratio=Hn/max_H),
    }

x0 = r0["x"]
ba = analyse_baselines(model, x0, r0["target"])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(denormalize(x0[0]).cpu().permute(1,2,0).numpy())
axes[0].set_title(f"Original\nP={r0['preds'][0][1]:.3f}", fontsize=10); axes[0].axis("off")

axes[1].imshow(denormalize(torch.zeros_like(x0)[0]).cpu().permute(1,2,0).numpy())
axes[1].set_title(f"Zero baseline\nP={ba['Zero']['p_target']:.4f} H/Hmax={ba['Zero']['entropy_ratio']:.2f}", fontsize=9)
axes[1].axis("off")

axes[2].imshow(denormalize(make_blur_baseline(x0)[0]).cpu().permute(1,2,0).numpy())
axes[2].set_title(f"Blur baseline\nP={ba['Blur']['p_target']:.4f} H/Hmax={ba['Blur']['entropy_ratio']:.2f}", fontsize=9)
axes[2].axis("off")

ns = torch.randn_like(x0)
axes[3].imshow(denormalize(ns[0]).cpu().permute(1,2,0).numpy().clip(0,1))
axes[3].set_title(f"N(0,1) sample\nE[P]={ba['N(0,1)']['p_target']:.4f} H/Hmax={ba['N(0,1)']['entropy_ratio']:.2f}", fontsize=9)
axes[3].axis("off")

fig.suptitle("Baseline Comparison: What Each Method Starts From", fontweight="bold")
plt.tight_layout(); plt.show()

print("\nIdeal reference: high H/Hmax (close to 1.0 = uniform), low P(target)")
for ref, vals in ba.items():
    print(f"  {ref:6s}: P(target)={vals['p_target']:.6f}  H/Hmax={vals['entropy_ratio']:.4f}")

---
## 14. Summary Dashboard

In [ ]:
# Aggregate all metrics across images
avg_ins = {m: np.mean([id_all[n][m]["ins_auc"] for n in id_all]) for m in methods}
avg_del = {m: np.mean([id_all[n][m]["del_auc"] for n in id_all]) for m in methods}
avg_comp = {m: np.mean([comp_all[n][m]["rel_err"]*100 for n in comp_all]) for m in methods}
avg_occ_r = {m: np.mean([occ_corrs[n][m] for n in occ_corrs]) for m in methods}
avg_gini_r = {m: np.mean(gini_all[m]) for m in methods}

summary = {
    "Ins. AUC":     (avg_ins,    "max"),
    "Del. AUC":     (avg_del,    "min"),
    "Compl. Err%":  (avg_comp,   "min"),
    "Occ. Corr":    (avg_occ_r,  "max"),
    "Gini":         (avg_gini_r, "max"),
}

# Print table
print("=" * 80)
print(f"{'METRIC':<16} {'Vanilla IG':>14} {'Blur IG':>14} {'KL-IG':>14} {'WINNER':>14}")
print("-" * 80)
for metric, (vals, direction) in summary.items():
    winner = max(vals, key=vals.get) if direction == "max" else min(vals, key=vals.get)
    print(f"{metric:<16} {vals['Vanilla IG']:>14.4f} {vals['Blur IG']:>14.4f} "
          f"{vals['KL-IG']:>14.4f} {'<-- '+winner:>14}")
print("=" * 80)

# Radar chart
import matplotlib.pyplot as plt
labels = list(summary.keys())
n = len(labels)
angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for method in methods:
    vals = []
    for metric, (d, direction) in summary.items():
        v = d[method]
        if direction == "min": v = 1.0 / (1.0 + v)
        vals.append(v)
    vals += vals[:1]
    ax.plot(angles, vals, "o-", lw=2, label=method, color=COLORS[method])
    ax.fill(angles, vals, alpha=0.08, color=COLORS[method])

ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, fontsize=9)
ax.set_title("KL-IG Evaluation Summary", fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
plt.tight_layout(); plt.show()